In [1]:
from pathlib import Path
import pandas as pd

RAW_FOLDER = Path("data/raw")

files = {
    "Bars.csv": "Bars",
    "Bookstores.csv": "Bookstores",
    "Cafés & Desserts.csv": "Cafes and Desserts",
    "Jazz.csv": "Jazz",
    "Museums & things.csv": "Museum & things",
    "Restaurants and Food.csv": "Restaurants and Food",
    "Things.csv": "Things",
    "Usable shops and markets.csv": "Usable Shops and Markets",
}

In [2]:
for filename, category in files.items():
    path = RAW_FOLDER / filename
    data = pd.read_csv(path)

    print(f"\n--- {category} ---")
    print("Columns:", data.columns.tolist())
    print(data.head(2))


--- Bars ---
Columns: ['Title', 'Note', 'URL', 'Tags', 'Comment']
     Title Note                                                URL  Tags  \
0      NaN  NaN                                                NaN   NaN   
1  Oddball  NaN  https://www.google.com/maps/place/Oddball/data...   NaN   

   Comment  
0      NaN  
1      NaN  

--- Bookstores ---
Columns: ['Title', 'Note', 'URL', 'Tags', 'Comment']
                           Title Note  \
0                            NaN  NaN   
1  Mercer Street Books & Records  NaN   

                                                 URL  Tags  Comment  
0                                                NaN   NaN      NaN  
1  https://www.google.com/maps/place/Mercer+Stree...   NaN      NaN  

--- Cafes and Desserts ---
Columns: ['Title', 'Note', 'URL', 'Tags', 'Comment']
                        Title Note  \
0                         NaN  NaN   
1  D'lioz Bakery ( HANDMADE )  NaN   

                                                 URL  Tags  Co

In [3]:
all_lists = []

expected_columns = ["Title", "Note", "URL", "Tags", "Comment"]

for filename, category in files.items():
    path = RAW_FOLDER / filename

    data = pd.read_csv(path, encoding="utf-8-sig")

    # Things.csv may contain an extra line before the real header
    if not set(expected_columns).issubset(data.columns):
        data = pd.read_csv(
            path,
            skiprows=1,
            encoding="utf-8-sig",
        )

    # Keep the relevant columns
    data = data[expected_columns].copy()

    # Remove completely blank rows
    data = data.dropna(
        subset=["Title", "URL"],
        how="all",
    )

    # Add the website category
    data["Category"] = category

    all_lists.append(data)

places = pd.concat(
    all_lists,
    ignore_index=True,
)

places.head()

,Title,Note,URL,Tags,Comment,Category
0,Oddball,NaN,https://www.google.com/maps/place/Oddball/data...,NaN,NaN,Bars
1,Public Records,NaN,https://www.google.com/maps/place/Public+Recor...,NaN,NaN,Bars
2,Double Chicken Please,NaN,https://www.google.com/maps/place/Double+Chick...,NaN,NaN,Bars
3,The Vale Public House NYC,NaN,https://www.google.com/maps/place/The+Vale+Pub...,NaN,NaN,Bars
4,Cafe Balearica,NaN,https://www.google.com/maps/place/Cafe+Baleari...,NaN,NaN,Bars


In [4]:
print("Total category entries:", len(places))

places["Category"].value_counts()

Total category entries: 482


Category
Restaurants and Food        146
Cafes and Desserts          121
Bars                         77
Things                       49
Museum & things              37
Jazz                         24
Bookstores                   22
Usable Shops and Markets      6
Name: count, dtype: int64

In [5]:
places["URL_clean"] = places["URL"].str.strip()

duplicate_places = places[places["URL_clean"].duplicated(keep=False)].sort_values(
    "URL_clean"
)

print("Total entries:", len(places))
print("Unique Google Maps places:", places["URL_clean"].nunique())
print("Entries belonging to multiple categories:", len(duplicate_places))

duplicate_places[["Title", "Category", "URL_clean"]].head(30)

Total entries: 482
Unique Google Maps places: 469
Entries belonging to multiple categories: 26


,Title,Category,URL_clean
62,Aubi & Ramsa,Bars,https://www.google.com/maps/place/Aubi+%26+Ram...
195,Aubi & Ramsa,Cafes and Desserts,https://www.google.com/maps/place/Aubi+%26+Ram...
63,Aubi & Ramsa,Bars,https://www.google.com/maps/place/Aubi+%26+Ram...
196,Aubi & Ramsa,Cafes and Desserts,https://www.google.com/maps/place/Aubi+%26+Ram...
58,Book Club Bar,Bars,https://www.google.com/maps/place/Book+Club+Ba...
81,Book Club Bar,Bookstores,https://www.google.com/maps/place/Book+Club+Ba...
136,Catania Bakery,Cafes and Desserts,https://www.google.com/maps/place/Catania+Bake...
306,Catania Bakery,Restaurants and Food,https://www.google.com/maps/place/Catania+Bake...
99,D'lioz Bakery ( HANDMADE ),Cafes and Desserts,https://www.google.com/maps/place/D'lioz+Baker...
284,D'lioz Bakery ( HANDMADE ),Restaurants and Food,https://www.google.com/maps/place/D'lioz+Baker...


In [6]:
def first_nonempty(series):
    values = series.dropna().astype(str).str.strip()
    values = values[values != ""]
    return values.iloc[0] if len(values) else ""


def combine_unique(series):
    values = series.dropna().astype(str).str.strip()
    values = values[values != ""]
    return " | ".join(dict.fromkeys(values))


category_order = list(dict.fromkeys(files.values()))


def combine_categories(series):
    present = set(series.dropna())
    return " | ".join(category for category in category_order if category in present)


master_places = (
    places.groupby("URL_clean", as_index=False)
    .agg(
        Title=("Title", first_nonempty),
        Note=("Note", combine_unique),
        Tags=("Tags", combine_unique),
        Comment=("Comment", combine_unique),
        Categories=("Category", combine_categories),
    )
    .rename(columns={"URL_clean": "URL"})
)

print("Master places:", len(master_places))

master_places[["Title", "Categories", "Note", "URL"]].head(20)

Master places: 469


,Title,Categories,Note,URL
0,222 Speakeasy,Bars,,https://www.google.com/maps/place/222+Speakeas...
1,308 E 8th St,Jazz,Live garden jazz,https://www.google.com/maps/place/308+E+8th+St...
2,4021 5th Ave,Restaurants and Food,Flor bakery,https://www.google.com/maps/place/4021+5th+Ave...
3,5ive Spice Taco & Banh Mi,Restaurants and Food,Banh mi,https://www.google.com/maps/place/5ive+Spice+T...
4,5ive Spice Tacos & Banh Mi,Restaurants and Food,Banh mi,https://www.google.com/maps/place/5ive+Spice+T...
5,69 Atlantic,Bars,Magic show on weekends,https://www.google.com/maps/place/69+Atlantic/...
6,6BC Botanical Garden,Things,,https://www.google.com/maps/place/6BC+Botanica...
7,787 Coffee,Cafes and Desserts,,https://www.google.com/maps/place/787+Coffee/d...
8,787 Coffee,Cafes and Desserts,,https://www.google.com/maps/place/787+Coffee/d...
9,81 St-Museum of Natural History,Museum & things,,https://www.google.com/maps/place/81+St-Museum...


In [7]:
OUTPUT_FOLDER = Path("data")
OUTPUT_FOLDER.mkdir(exist_ok=True)

master_places["Status"] = "visible"
master_places["Featured"] = "no"

master_places.to_csv(
    OUTPUT_FOLDER / "places-master.csv",
    index=False,
)

print("Saved:", OUTPUT_FOLDER / "places-master.csv")
print("Rows:", len(master_places))

Saved: data/places-master.csv
Rows: 469


In [8]:
import re


def extract_coordinates(url):
    if pd.isna(url):
        return pd.Series([None, None])

    # Pattern 1: /@40.7128,-74.0060
    match = re.search(
        r"/@(-?\d+\.\d+),(-?\d+\.\d+)",
        url,
    )

    if match:
        return pd.Series(
            [
                float(match.group(1)),  # latitude
                float(match.group(2)),  # longitude
            ]
        )

    # Pattern 2: !3d40.7128!4d-74.0060
    match = re.search(
        r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)",
        url,
    )

    if match:
        return pd.Series(
            [
                float(match.group(1)),
                float(match.group(2)),
            ]
        )

    return pd.Series([None, None])


master_places[["Latitude", "Longitude"]] = master_places["URL"].apply(
    extract_coordinates
)

print(
    "Places with coordinates:",
    master_places["Latitude"].notna().sum(),
)

print(
    "Places still missing coordinates:",
    master_places["Latitude"].isna().sum(),
)

master_places.to_csv(
    OUTPUT_FOLDER / "places-master.csv",
    index=False,
)

Places with coordinates: 0
Places still missing coordinates: 469


In [9]:
pd.set_option("display.max_colwidth", None)

print(master_places.loc[0, "URL"])
print()
print(master_places.loc[1, "URL"])

https://www.google.com/maps/place/222+Speakeasy/data=!4m2!3m1!1s0x89c25953732ce725:0x67738d3649b2184f

https://www.google.com/maps/place/308+E+8th+St/data=!4m2!3m1!1s0x89c259777e358acf:0xdf7b5d2344d971fd


In [1]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="krittika-nyc-recommendations-map")

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1,
    swallow_exceptions=True,
)

In [8]:
from pathlib import Path
import time
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

DATA_FOLDER = Path("data")
PROGRESS_FILE = DATA_FOLDER / "geocoding-progress.csv"

# Resume saved progress if this cell was previously interrupted
if PROGRESS_FILE.exists():
    master_places = pd.read_csv(PROGRESS_FILE)
    print("Resuming saved progress.")
else:
    master_places = pd.read_csv(DATA_FOLDER / "places-master.csv")

for column in ["Latitude", "Longitude", "Matched_Address"]:
    if column not in master_places.columns:
        master_places[column] = pd.NA

geolocator = Nominatim(
    user_agent="krittika-nyc-recommendations-map/1.0",
    timeout=15,
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.1,
    swallow_exceptions=True,
)

# Approximate bounds of New York City
nyc_viewbox = [
    (40.49, -74.27),
    (40.92, -73.68),
]

remaining = master_places["Latitude"].isna().sum()
print(f"Places remaining: {remaining}")

for count, index in enumerate(
    master_places[master_places["Latitude"].isna()].index,
    start=1,
):
    title = master_places.at[index, "Title"]

    location = geocode(
        f"{title}, New York City, New York, USA",
        country_codes="us",
        viewbox=nyc_viewbox,
        bounded=True,
        exactly_one=True,
    )

    if location is not None:
        master_places.at[index, "Latitude"] = location.latitude
        master_places.at[index, "Longitude"] = location.longitude
        master_places.at[index, "Matched_Address"] = location.address

    # Save progress every 10 requests
    if count % 10 == 0:
        master_places.to_csv(PROGRESS_FILE, index=False)
        print(f"Processed {count} of {remaining}")

# Save final results
master_places.to_csv(PROGRESS_FILE, index=False)

matched_places = master_places[master_places["Latitude"].notna()].copy()

unmatched_places = master_places[master_places["Latitude"].isna()].copy()

matched_places.to_csv(
    DATA_FOLDER / "places-geocoded.csv",
    index=False,
)

unmatched_places.to_csv(
    DATA_FOLDER / "places-unmatched.csv",
    index=False,
)

print()
print("Finished.")
print("Matched:", len(matched_places))
print("Unmatched:", len(unmatched_places))

Places remaining: 469
Processed 10 of 469
Processed 20 of 469
Processed 30 of 469
Processed 40 of 469
Processed 50 of 469
Processed 60 of 469
Processed 70 of 469
Processed 80 of 469
Processed 90 of 469
Processed 100 of 469
Processed 110 of 469
Processed 120 of 469
Processed 130 of 469
Processed 140 of 469
Processed 150 of 469
Processed 160 of 469
Processed 170 of 469
Processed 180 of 469
Processed 190 of 469
Processed 200 of 469
Processed 210 of 469
Processed 220 of 469
Processed 230 of 469
Processed 240 of 469
Processed 250 of 469
Processed 260 of 469
Processed 270 of 469
Processed 280 of 469
Processed 290 of 469
Processed 300 of 469
Processed 310 of 469
Processed 320 of 469
Processed 330 of 469
Processed 340 of 469
Processed 350 of 469
Processed 360 of 469
Processed 370 of 469
Processed 380 of 469
Processed 390 of 469
Processed 400 of 469
Processed 410 of 469
Processed 420 of 469
Processed 430 of 469
Processed 440 of 469
Processed 450 of 469
Processed 460 of 469

Finished.
Matched: 3

In [9]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

DATA_FOLDER = Path("data")

matched_places = pd.read_csv(DATA_FOLDER / "places-geocoded.csv")

recommendations = gpd.GeoDataFrame(
    matched_places,
    geometry=gpd.points_from_xy(
        matched_places["Longitude"],
        matched_places["Latitude"],
    ),
    crs="EPSG:4326",
)

recommendations.to_file(
    DATA_FOLDER / "nyc-recommendations.geojson",
    driver="GeoJSON",
)

print("GeoJSON places:", len(recommendations))
print("Saved: data/nyc-recommendations.geojson")

GeoJSON places: 308
Saved: data/nyc-recommendations.geojson


In [10]:
nta_url = "https://data.cityofnewyork.us/resource/" "9nt8-h7nd.geojson?$limit=500"

neighborhoods = gpd.read_file(nta_url).to_crs("EPSG:4326")

recommendations = gpd.sjoin(
    recommendations,
    neighborhoods[["ntaname", "boroname", "geometry"]],
    how="left",
    predicate="within",
)

recommendations = recommendations.rename(
    columns={
        "ntaname": "Neighborhood",
        "boroname": "Borough",
    }
)

recommendations = recommendations.drop(
    columns=["index_right"],
    errors="ignore",
)

In [11]:
emoji_map = {
    "Restaurants and Food": "🍴",
    "Bars": "🍷",
    "Things": "✨",
    "Cafes and Desserts": "☕",
    "Museum & things": "🏛️",
    "Jazz": "🎷",
    "Usable Shops and Markets": "🛍️",
    "Bookstores": "📚",
}

recommendations["Primary_Category"] = (
    recommendations["Categories"].str.split("|").str[0].str.strip()
)

recommendations["Emoji"] = (
    recommendations["Primary_Category"].map(emoji_map).fillna("📍")
)

recommendations.to_file(
    DATA_FOLDER / "nyc-recommendations.geojson",
    driver="GeoJSON",
)